# 13. NJ feature experiments and report diagnostics

Validation-only comparison of daily recorded-onset forecasts for horizons 1, 2 and 3. All models use the same 590 NJ regions and 362 forecast dates in 2021. Weekly refitting uses 30 days as the primary window and 60 days as sensitivity. Earlier test results have already been inspected; these results are exploratory validation, not independent final confirmation.

## Feature specifications

All specifications include region fixed effects and target weekday indicators.

| ID | Features added |
|---|---|
| B0 | Today, yesterday and lag-7 case counts |
| H0 | B0 + trailing 7-day mean |
| H1 | H0 + lag-14 count and trailing 7-day standard deviation |
| D1 | H1 + acceleration |
| D2 | D1 + jerk |
| P0 | B0 + 7-day commuting pressure |
| P1 | D2 + 7-day commuting pressure |
| P2 | D2 + 14-day commuting pressure |

Velocity is exactly redundant with today and yesterday. Acceleration and jerk expand the effective lag basis; they are not independent epidemiological measurements. Static regional attributes are absorbed by region fixed effects. P1/P2 test pressure on D2, not on H0.

## Statistical records

* Prediction comparisons: paired daily mean losses, HAC standard errors (14-day bandwidth), Holm correction across 64 contrasts, and 2,000 moving-date-block bootstrap replicates. Positive delta MAE means improvement. Bootstrap CIs are pointwise and resample fixed forecast errors, not fitted models. Block lengths 7/28 provide sensitivity checks.
* Coefficients: approximate region/date clustered standard errors, normal-reference p-values and intervals for each weekly refit. Dynamic FE bias and endogeneity are not corrected. Interpret as exploratory conditional associations, not causal effects. Do not average overlapping-refit p-values.
* Sargan/Hansen and Arellano–Bond AR(2) are not applicable to this FE OLS estimator. PR-AUC, warning lead time and spatial Moran diagnostics require subsequent outbreak definitions and spatial weights.

The outcome remains retrospectively recorded illness-onset rows. Upstream missing-onset and ZIP–ZCTA mapping limitations remain.


In [1]:
"""Validation-only feature ablations and exploratory inference for NJ onset counts.

Requires numpy/pandas. Optional matplotlib creates report figures. No GMM tests.
"""
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import platform
import numpy as np
import pandas as pd

REPO=Path(r'C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun')
SOURCE=REPO/'nj_daily_feature_outputs'/'nj_daily_features.csv.gz'

BASE=['cases_today','cases_lag_1','cases_lag_7']
HISTORY=BASE+['cases_mean_7d']
EXTENDED=HISTORY+['cases_lag_14','case_std_7d']
ACC=EXTENDED+['case_acceleration']
DYNAMICS=ACC+['case_jerk']
SPECIFICATIONS={
    'B0_basic':BASE,
    'H0_weekly_mean':HISTORY,
    'H1_history_extended':EXTENDED,
    'D1_acceleration':ACC,
    'D2_jerk':DYNAMICS,
    'P0_basic_pressure7':BASE+['nj_covid_pressure_7d'],
    'P1_rich_pressure7':DYNAMICS+['nj_covid_pressure_7d'],
    'P2_rich_pressure14':DYNAMICS+['nj_covid_pressure_14d'],
}
COMPARISONS=[
    ('B0_basic','H0_weekly_mean','weekly_mean'),
    ('H0_weekly_mean','H1_history_extended','lag14_and_volatility'),
    ('H1_history_extended','D1_acceleration','acceleration'),
    ('D1_acceleration','D2_jerk','jerk'),
    ('B0_basic','P0_basic_pressure7','pressure7_basic'),
    ('D2_jerk','P1_rich_pressure7','pressure7_rich'),
    ('D2_jerk','P2_rich_pressure14','pressure14_rich'),
    ('P1_rich_pressure7','P2_rich_pressure14','pressure14_vs_pressure7'),
]


def two_way_meat(scores,n_parameters):
    """CR1 two-way cluster sandwich meat: region + date - region/date cell."""
    N,T,p=scores.shape
    n=N*T
    if min(N,T)<=1 or n<=n_parameters:
        return np.full((p,p),np.nan)
    base=(n-1)/(n-n_parameters)
    by_region=scores.sum(axis=1)
    by_date=scores.sum(axis=0)
    cells=scores.reshape(-1,p)
    return base*(N/(N-1)*(by_region.T@by_region)+T/(T-1)*(by_date.T@by_date)-n/(n-1)*(cells.T@cells))


def fit_fe(x,y):
    N,T,p=x.shape
    assert np.isfinite(x).all() and np.isfinite(y).all()
    xm=x.mean(axis=1); ym=y.mean(axis=1)
    xc=x-xm[:,None,:]; yc=y-ym[:,None]
    scale=np.sqrt(np.mean(xc*xc,axis=(0,1)))
    scale=np.where(scale>1e-12,scale,1)
    z=xc/scale
    zf=z.reshape(-1,p)
    b,_,rank,sv=np.linalg.lstsq(zf,yc.reshape(-1),rcond=None)
    beta=b/scale; alpha=ym-xm@beta
    residual=yc-np.einsum('ntp,p->nt',z,b)
    if rank==p:
        bread=np.linalg.pinv(zf.T@zf,rcond=1e-12)
        covariance=bread@two_way_meat(z*residual[:,:,None],N+p)@bread
        variance=np.diag(covariance)/(scale*scale)
        se=np.full(p,np.nan)
        positive=np.isfinite(variance)&(variance>0)
        se[positive]=np.sqrt(variance[positive])
    else:
        se=np.full(p,np.nan)
    pvalue=np.array([math.erfc(abs(v/s)/math.sqrt(2)) if np.isfinite(s) and s>0 else np.nan for v,s in zip(beta,se)])
    condition=float(sv[0]/sv[-1]) if len(sv) and sv[-1]>0 else np.inf
    return beta,alpha,se,pvalue,int(rank),condition


def hac_mean_test(d,bandwidth=14):
    """Two-sided asymptotic test of zero mean paired DATE-level loss difference."""
    d=np.asarray(d,float)
    assert np.isfinite(d).all()
    T=len(d); L=min(int(bandwidth),T-1)
    mean=float(d.mean()); c=d-mean
    longvar=float(c@c/T)
    for lag in range(1,L+1):
        longvar+=2*(1-lag/(L+1))*float(c[lag:]@c[:-lag]/T)
    variance=longvar/T*T/(T-1)
    if variance<=0 or not np.isfinite(variance):
        return mean,np.nan,np.nan
    se=math.sqrt(variance)
    return mean,se,math.erfc(abs(mean/se)/math.sqrt(2))


def holm_adjust(p):
    p=np.asarray(p,float); adjusted=np.full(p.shape,np.nan)
    good=np.flatnonzero(np.isfinite(p)); ordered=good[np.argsort(p[good])]
    if len(ordered):
        values=np.maximum.accumulate((len(ordered)-np.arange(len(ordered)))*p[ordered])
        adjusted[ordered]=np.minimum(values,1)
    return adjusted


def block_indices(T,block_length,repetitions,seed):
    """Noncircular moving-block resampling; one date index shared by all regions/horizons."""
    assert 1<=block_length<=T
    rng=np.random.default_rng(seed)
    starts=rng.integers(0,T-block_length+1,size=(repetitions,math.ceil(T/block_length)))
    return (starts[:,:,None]+np.arange(block_length)[None,None,:]).reshape(repetitions,-1)[:,:T]


def residual_correlation(errors,lag):
    a=errors[:,lag:]; b=errors[:,:-lag]
    a=a-a.mean(axis=1,keepdims=True); b=b-b.mean(axis=1,keepdims=True)
    den=np.sqrt(np.sum(a*a,axis=1)*np.sum(b*b,axis=1))
    value=np.full(len(a),np.nan)
    good=den>0; value[good]=np.sum(a[good]*b[good],axis=1)/den[good]
    return value


def self_checks():
    # FWL estimates and cluster covariance versus an explicit dummy design.
    rng=np.random.default_rng(729)
    N,T,p=5,12,2
    x=rng.normal(size=(N,T,p)); y=x@np.array([.5,-.25])+np.arange(N)[:,None]+rng.normal(size=(N,T))
    beta,alpha,se,pv,rank,_=fit_fe(x,y)
    design=np.column_stack([np.repeat(np.eye(N),T,axis=0),x.reshape(-1,p)])
    full=np.linalg.lstsq(design,y.reshape(-1),rcond=None)[0]
    assert np.allclose(full[:N],alpha) and np.allclose(full[N:],beta)
    e=y.reshape(-1)-design@full
    inv=np.linalg.pinv(design.T@design)
    covariance=inv@two_way_meat((design*e[:,None]).reshape(N,T,N+p),N+p)@inv
    positive=np.diag(covariance)[N:]>0
    assert np.allclose(se[positive]**2,np.diag(covariance)[N:][positive])
    assert np.allclose(holm_adjust([.01,.04,.03]),[.03,.06,.06])
    independent=rng.normal(size=100)
    _,s,_=hac_mean_test(independent,0)
    assert np.isclose(s,np.std(independent,ddof=1)/10)


def run_feature_experiments(source=SOURCE,output=None,windows=(30,60),retrain_every=7,
                            validation_start='2021-01-01',validation_end='2021-12-31',
                            bootstrap_repetitions=2000,save_predictions=True):
    self_checks()
    source=Path(source)
    if output is None:output=source.parent.parent/'dynamic_panel_feature_experiment_outputs'
    folder=Path(output)/('feature_ablation_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ'))
    folder.mkdir(parents=True)
    cols=sorted(set(['zip','date','cases_today','pressure_source_weight_coverage_nj_7d','pressure_source_weight_coverage_nj_14d','pressure_definition']+sum(SPECIFICATIONS.values(),[])))
    panel=pd.read_csv(source,usecols=cols,dtype={'zip':'string'},parse_dates=['date'])
    assert panel.pressure_definition.eq('valid_population_nj_origins_renormalized_v2').all()
    assert not panel.duplicated(['zip','date']).any()
    all_ids=sorted(panel.zip.unique()); dates=pd.date_range(panel.date.min(),panel.date.max())
    assert len(panel)==len(all_ids)*len(dates)
    def matrix(c):
        return panel.pivot(index='zip',columns='date',values=c).reindex(index=all_ids,columns=dates).to_numpy(dtype=float)
    matrices={c:matrix(c) for c in cols if c not in ['zip','date','pressure_definition']}
    # Fixed source-availability cohort, determined before evaluation, not by errors.
    eligible=np.isfinite(matrices['nj_covid_pressure_7d'][:,13])&np.isfinite(matrices['nj_covid_pressure_14d'][:,13])
    for w in [7,14]:
        eligible &= np.isclose(matrices[f'pressure_source_weight_coverage_nj_{w}d'][:,13],1,atol=1e-10,rtol=0)
    ids=np.array(all_ids)[eligible]
    assert len(ids)>0
    matrices={c:v[eligible] for c,v in matrices.items()}
    N=len(ids); counts=matrices['cases_today']; T=len(dates)
    assert np.isfinite(counts).all() and (counts>=0).all()
    start,end=pd.Timestamp(validation_start),pd.Timestamp(validation_end)
    assert dates.min()<start<=end<=dates.max()
    origins=np.flatnonzero((dates>=start)&((dates+pd.Timedelta(days=3))<=end))
    assert len(origins)>28 and np.all(np.diff(origins)==1)
    eval_dates=dates[origins]; V=len(origins)
    windows=tuple(sorted(set(windows)))
    assert set(windows)=={30,60}, 'This prespecified experiment uses windows 30 and 60; revise protocol explicitly for other windows.'
    # Descriptive volume groups use only history before validation.
    volume=counts[:,dates<start].mean(axis=1)
    pct=pd.Series(volume).rank(pct=True,method='average')
    volume_group=pd.cut(pct,[0,.25,.5,.75,1],labels=['low','lower_middle','upper_middle','high']).astype(str).to_numpy()
    pd.DataFrame({'zip':ids,'pre_validation_mean_cases':volume,'pre_validation_volume_group':volume_group}).to_csv(folder/'region_cohort.csv',index=False)
    config={'source':str(source.resolve()),'source_sha256':hashlib.sha256(source.read_bytes()).hexdigest(),
        'windows':list(windows),'primary_window':30,'sensitivity_window':60,'horizons':[1,2,3],
        'validation_start':validation_start,'validation_end':validation_end,'regions':N,'forecast_dates':V,
        'retrain_every_days':retrain_every,'specifications':SPECIFICATIONS,'comparisons':COMPARISONS,
        'primary_metric':'Equal-weight mean validation MAE across horizons 1/2/3.',
        'prediction_test':'Two-sided normal-approximation HAC test of mean daily paired absolute-loss differences; Bartlett bandwidth 14; 7/28 sensitivity.',
        'multiplicity':'Holm adjustment over all planned comparisons x overall/horizons x both windows (64 tests).',
        'bootstrap':'Paired noncircular moving-date-block percentile intervals; 14-day primary, 7/28 sensitivity; fixed saved forecasts, no refitting in resamples.',
        'bootstrap_repetitions':bootstrap_repetitions,'random_seed_base':20261026,
        'coefficient_inference':'Exploratory normal-approximate two-way region/date clustered CR1 SE and Wald p-values for FE OLS; does not correct dynamic-panel finite-T bias.',
        'test_evaluated':False,'inference_status':'Exploratory on repeatedly inspected validation data; not confirmatory, not causal, no post-selection-valid inference claimed.',
        'outcome':'Final retrospective recorded-onset row count; missing onset excluded upstream; absent rows zero by user rule.',
        'geography':'Current exact ZIP/ZCTA identifier match remains unvalidated; pressure uses static 2021 population and commuting assumptions.',
        'versions':{'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__}}
    (folder/'experiment_protocol.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    pd.DataFrame([{'model':m,'feature_list':'; '.join(f),'weekday_dummies':6,'region_fixed_effects':True} for m,f in SPECIFICATIONS.items()]).to_csv(folder/'model_specifications.csv',index=False)
    pd.DataFrame([
        ('Sargan / Hansen instruments','not applicable','No GMM estimator or instruments in this experiment.'),
        ('Arellano-Bond AR(2)','not applicable','Not the GMM differenced-residual diagnostic; save descriptive forecast residual correlations instead.'),
        ('PR-AUC / warning lead time','deferred','Requires an agreed outbreak event and warning rule.'),
        ('Residual Moran I','deferred','Requires an agreed real geographic weights matrix; no random substitute.')
    ],columns=['diagnostic','status','reason']).to_csv(folder/'diagnostic_scope.csv',index=False)
    metrics=[]; period_metrics=[]; region_metrics=[]; daily_losses=[]; fit_log=[]; coefficients=[]; comparison_rows=[]; interval_rows=[]
    models=list(SPECIFICATIONS)
    weekday_names=[f'target_weekday_{k}' for k in range(2,8)]
    for window in windows:
        print(f'Window {window}: {len(models)} models, {N} common regions, validation only.',flush=True)
        actual=np.stack([counts[:,origins+h] for h in [1,2,3]])
        raw_predictions=np.full((len(models),3,N,V),np.nan)
        for h in [1,2,3]:
            weekday=np.eye(7)[(dates+pd.Timedelta(days=h)).dayofweek.to_numpy(),1:]
            for m,model in enumerate(models):
                feature_names=SPECIFICATIONS[model]+weekday_names
                x=np.concatenate([np.stack([matrices[c] for c in SPECIFICATIONS[model]],axis=2),np.broadcast_to(weekday,(N,T,6))],axis=2)
                for first in range(0,V,retrain_every):
                    block=origins[first:first+retrain_every]
                    fit_i=int(block[0]); train_end=fit_i-h; train_start=train_end-window+1
                    assert train_start>=14 and train_end+h<=fit_i
                    train=np.arange(train_start,train_end+1)
                    beta,alpha,se,pvals,rank,condition=fit_fe(x[:,train],counts[:,train+h])
                    pred=np.einsum('ntp,p->nt',x[:,block],beta)+alpha[:,None]
                    raw_predictions[m,h-1,:,first:first+len(block)]=pred
                    fit_log.append({'window':window,'horizon':h,'model':model,'fit_date':dates[fit_i],
                        'training_origin_start':dates[train_start],'training_origin_end':dates[train_end],
                        'latest_training_target':dates[train_end+h],'training_rows':N*window,
                        'prediction_origin_end':dates[block[-1]],'regions':N,'features':len(beta),
                        'design_rank':rank,'scaled_design_condition_number':condition,'raw_negative_predictions':int((pred<0).sum()),
                        'coefficient_se_unavailable_count':int(np.isnan(se).sum())})
                    for name,b,s,pv in zip(feature_names,beta,se,pvals):
                        coefficients.append({'window':window,'horizon':h,'model':model,'fit_date':dates[fit_i],
                            'feature':name,'estimate':b,'se_two_way_cluster_approx':s,
                            'ci95_low_approx':b-1.95996398454*s,'ci95_high_approx':b+1.95996398454*s,
                            'p_approx_exploratory':pv,'inference_status':'exploratory_dynamic_FE_OLS_no_causal_claim',
                            'se_status':'available' if np.isfinite(s) else 'rank_deficient_or_nonpositive_cluster_variance'})
                print(f'  h={h}: {model} fitted.',flush=True)
        assert np.isfinite(raw_predictions).all()
        pred=np.maximum(raw_predictions,0)
        error=pred-actual[None,:,:,:]
        daily_ae=np.abs(error).mean(axis=2)  # model x horizon x forecast date
        daily_mse=(error*error).mean(axis=2)
        daily_bias=error.mean(axis=2)
        quarter=np.array([f'{d.year}Q{d.quarter}' for d in eval_dates])
        for m,model in enumerate(models):
            for hh in range(3):
                e=error[m,hh]
                metrics.append({'window':window,'model':model,'horizon':hh+1,'regions':N,'dates':V,'rows':N*V,
                    'MAE':float(daily_ae[m,hh].mean()),'RMSE':float(np.sqrt(daily_mse[m,hh].mean())),
                    'bias':float(e.mean()),'mean_actual':float(actual[hh].mean()),
                    'median_region_residual_lag1_corr':float(np.nanmedian(residual_correlation(e,1))),
                    'median_region_residual_lag7_corr':float(np.nanmedian(residual_correlation(e,7)))})
                for t,d in enumerate(eval_dates):
                    daily_losses.append({'window':window,'model':model,'horizon':hh+1,'date':d,
                        'MAE':daily_ae[m,hh,t],'MSE':daily_mse[m,hh,t],'bias':daily_bias[m,hh,t]})
                for q in sorted(set(quarter)):
                    mask=quarter==q; eq=e[:,mask]
                    period_metrics.append({'window':window,'model':model,'horizon':hh+1,'quarter':q,
                        'rows':int(eq.size),'MAE':float(np.abs(eq).mean()),'RMSE':float(np.sqrt(np.mean(eq*eq))),'bias':float(eq.mean())})
                for i,zz in enumerate(ids):
                    region_metrics.append({'window':window,'model':model,'horizon':hh+1,'zip':zz,
                        'pre_validation_volume_group':volume_group[i],'MAE':float(np.abs(e[i]).mean()),
                        'RMSE':float(np.sqrt(np.mean(e[i]*e[i]))),'bias':float(e[i].mean())})
        if save_predictions:
            dest=folder/'predictions';dest.mkdir(exist_ok=True)
            np.savez_compressed(dest/f'window_{window}d.npz',raw_predictions=raw_predictions,
                actual_cases=actual,models=np.array(models),zip=ids,date=eval_dates.to_numpy(),horizons=np.array([1,2,3]))
        # Joint date-block indices preserve paired forecasts and cross-region dependence.
        draws={b:block_indices(V,b,bootstrap_repetitions,20261026+window+b) for b in [7,14,28]}
        for baseline,candidate,label in COMPARISONS:
            a=models.index(baseline); b=models.index(candidate)
            for horizon in ['overall',1,2,3]:
                hs=np.arange(3) if horizon=='overall' else np.array([int(horizon)-1])
                daily_difference=(daily_ae[a,hs]-daily_ae[b,hs]).mean(axis=0)
                delta,se,pvalue=hac_mean_test(daily_difference,14)
                base_mae=float(daily_ae[a,hs].mean()); candidate_mae=float(daily_ae[b,hs].mean())
                base_rmse=float(np.sqrt(daily_mse[a,hs].mean(axis=1)).mean())
                candidate_rmse=float(np.sqrt(daily_mse[b,hs].mean(axis=1)).mean())
                row={'window':window,'comparison':label,'baseline':baseline,'candidate':candidate,'horizon':horizon,
                    'dates':V,'regions':N,'baseline_MAE':base_mae,'candidate_MAE':candidate_mae,
                    'delta_MAE_baseline_minus_candidate':delta,'MAE_reduction_percent':delta/base_mae*100,
                    'baseline_mean_RMSE':base_rmse,'candidate_mean_RMSE':candidate_rmse,
                    'delta_mean_RMSE':base_rmse-candidate_rmse,'daily_loss_HAC_SE':se,'p_HAC_two_sided_raw':pvalue,
                    'p_HAC_bandwidth7':hac_mean_test(daily_difference,7)[2],
                    'p_HAC_bandwidth28':hac_mean_test(daily_difference,28)[2],
                    'region_share_with_lower_MAE':float((np.abs(error[b,hs]).mean(axis=(0,2))<np.abs(error[a,hs]).mean(axis=(0,2))).mean()),
                    'inference_status':'exploratory_validation_reused_not_confirmatory'}
                for length,index in draws.items():
                    boot_delta=daily_difference[index].mean(axis=1)
                    # Average of horizon RMSEs, not pooled RMSE across horizons.
                    boot_rmse=np.sqrt(daily_mse[a,hs][:,index].mean(axis=2)).mean(axis=0)-np.sqrt(daily_mse[b,hs][:,index].mean(axis=2)).mean(axis=0)
                    lo,hi=np.quantile(boot_delta,[.025,.975])
                    rlo,rhi=np.quantile(boot_rmse,[.025,.975])
                    interval_rows.append({'window':window,'comparison':label,'horizon':horizon,'block_days':length,
                        'bootstrap_repetitions':bootstrap_repetitions,'delta_MAE_ci95_low':float(lo),'delta_MAE_ci95_high':float(hi),
                        'delta_mean_RMSE_ci95_low':float(rlo),'delta_mean_RMSE_ci95_high':float(rhi),
                        'interval_scope':'pointwise_percentile_exploratory_fixed_saved_forecasts'})
                    if length==14:
                        row.update(delta_MAE_ci95_low=float(lo),delta_MAE_ci95_high=float(hi),
                            delta_mean_RMSE_ci95_low=float(rlo),delta_mean_RMSE_ci95_high=float(rhi))
                comparison_rows.append(row)
        print(f'Window {window} completed; paired loss comparisons recorded.',flush=True)
    metrics=pd.DataFrame(metrics); comparisons=pd.DataFrame(comparison_rows)
    comparisons['p_HAC_Holm_all_64']=holm_adjust(comparisons.p_HAC_two_sided_raw)
    comparisons['positive_MAE_improvement']=comparisons.delta_MAE_baseline_minus_candidate.gt(0)
    comparisons['Holm_0_05_and_positive']=comparisons.p_HAC_Holm_all_64.lt(.05)&comparisons.positive_MAE_improvement
    rankings=metrics.groupby(['window','model'],as_index=False).agg(mean_MAE=('MAE','mean'),mean_RMSE=('RMSE','mean'),mean_bias=('bias','mean'))
    rankings=rankings.sort_values(['window','mean_MAE','mean_RMSE'])
    rankings['rank_within_window']=rankings.groupby('window').cumcount()+1
    logs=pd.DataFrame(fit_log)
    assert logs.latest_training_target.le(logs.fit_date).all()
    assert len(comparisons)==64
    assert np.allclose(comparisons.baseline_MAE-comparisons.candidate_MAE,comparisons.delta_MAE_baseline_minus_candidate)
    metrics.to_csv(folder/'metrics_by_model_horizon.csv',index=False)
    rankings.to_csv(folder/'model_ranking.csv',index=False)
    comparisons.to_csv(folder/'paired_prediction_tests.csv',index=False)
    pd.DataFrame(interval_rows).to_csv(folder/'block_length_sensitivity.csv',index=False)
    pd.DataFrame(period_metrics).to_csv(folder/'metrics_by_quarter.csv',index=False)
    pd.DataFrame(region_metrics).to_csv(folder/'metrics_by_region.csv',index=False)
    pd.DataFrame(daily_losses).to_csv(folder/'daily_losses.csv',index=False)
    logs.to_csv(folder/'training_log.csv',index=False)
    coef=pd.DataFrame(coefficients)
    coef.to_csv(folder/'coefficients_exploratory.csv',index=False)
    coef.groupby(['window','model','horizon','feature'],as_index=False).agg(
        median_estimate=('estimate','median'),estimate_q25=('estimate',lambda v:v.quantile(.25)),
        estimate_q75=('estimate',lambda v:v.quantile(.75)),positive_estimate_fraction=('estimate',lambda v:(v>0).mean()),
        refits=('estimate','size')).to_csv(folder/'coefficient_stability.csv',index=False)
    methods='''# Methods and reporting guide

Primary comparison window is 30 days; 60 days is sensitivity. All specifications use the same fixed NJ cohort, validation origins, direct horizons 1/2/3 and weekly refitting. Model variants and eight planned comparisons were specified before this run. This is a feature-ablation experiment, not a final test or a causal study.

`delta_MAE_baseline_minus_candidate > 0` means the candidate improves prediction. Daily loss first averages over all common regions (and over horizons for the overall comparison). A two-sided HAC mean-loss test uses Bartlett bandwidth 14, with 7/28 sensitivity. All 64 planned p-values are Holm-adjusted together. The 95% moving-date-block bootstrap intervals are pointwise, not simultaneous; they resample saved forecast errors, not model fits. Every region/horizon shares the same sampled dates. Approximation quality depends on temporal dependence and stability; COVID regime shifts and repeatedly inspected validation data limit confirmatory interpretation. No per-row independent t-test is used.

Coefficient SEs cluster by region and training date using CR1 and an asymptotic normal reference, with about 30 or 60 date clusters. They are exploratory conditional linear-model calculations. Dynamic FE OLS lagged-outcome bias, endogeneity, short windows and specification selection are NOT corrected. Invalid cluster variances or rank deficiency yield unavailable SE/p-values, not fabricated significance. Repeated refits overlap: do not average coefficient p-values or treat refits as independent evidence. Use coefficient stability descriptively. A small coefficient p-value is not proof of forecast improvement or causal transmission.

Acceleration is C(t)-2C(t-1)+C(t-2); jerk additionally involves C(t-3). With C(t) and C(t-1) already included, these expand the effective lag basis, not independent epidemiological measurements. Velocity is omitted because it is exactly redundant with the basic model. Mean and sum of the same window are not both included. Static population/density coefficients are not identified separately from full region fixed effects.

Sargan/Hansen and Arellano-Bond AR(2) are not applicable: no GMM estimator/instruments are fitted here. Residual lag correlations are descriptive and not a substitute for those tests. Moran, PR-AUC and lead time are deferred until appropriate spatial weights and outbreak definitions exist. In-sample R-squared is not used to select features.

The outcome is retrospectively recorded onset rows, not all infections. Unknown onset records and unvalidated ZIP/ZCTA mapping remain upstream limitations. A favorable validation result is a candidate for further temporal validation, not final independent proof. Earlier 2022-2023 test results have already been inspected. Use a new untouched period or a prespecified nested temporal assessment for stronger final performance claims.

Reference methods: statsmodels two-way cluster covariance, HAC covariance and multiple-testing documentation. Implementations here use numpy/pandas and Python math; no statsmodels dependency is required.

- https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_cluster_2groups.html
- https://www.statsmodels.org/stable/generated/statsmodels.stats.sandwich_covariance.cov_hac.html
- https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html
'''
    (folder/'METHODS.md').write_text(methods,encoding='utf-8')
    report='# NJ feature experiments: validation results\n\n'
    report+='Mean MAE/RMSE average the three horizon-specific scores. Lower is better. This is exploratory validation, not final holdout inference.\n\n'
    report+='| Window | Model | Mean MAE | Mean RMSE | Rank |\n|---|---|---:|---:|---:|\n'
    for r in rankings.itertuples():report+=f'| {r.window} | {r.model} | {r.mean_MAE:.6f} | {r.mean_RMSE:.6f} | {r.rank_within_window} |\n'
    report+='\n## Planned overall comparisons, primary 30-day window\n\nPositive delta means improvement. CIs are pointwise date-block bootstrap intervals. P-values are exploratory HAC tests adjusted across all 64 planned contrasts.\n\n'
    report+='| Comparison | Delta MAE | 95% CI | Raw p | Holm p |\n|---|---:|---|---:|---:|\n'
    for r in comparisons.loc[comparisons.window.eq(30)&comparisons.horizon.eq('overall')].itertuples():
        report+=f'| {r.comparison} | {r.delta_MAE_baseline_minus_candidate:.6f} | [{r.delta_MAE_ci95_low:.6f}, {r.delta_MAE_ci95_high:.6f}] | {r.p_HAC_two_sided_raw:.4g} | {r.p_HAC_Holm_all_64:.4g} |\n'
    report+='\nInspect metrics_by_quarter, individual horizons, bias, regional distribution and the 60-day sensitivity before retaining a feature. Do not select solely by p<0.05. See METHODS.md for interpretation and inference limits.\n'
    (folder/'RESULTS.md').write_text(report,encoding='utf-8')
    summary={'rows_source_panel':len(panel),'matched_regions':N,'validation_forecast_dates':V,'models':len(models),
        'training_windows':list(windows),'model_fits':len(logs),'paired_tests':len(comparisons),
        'test_evaluated':False,'self_checks_passed':True,'path':str(folder.resolve())}
    (folder/'run_summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(rankings.to_string(index=False),flush=True)
    print('Saved to:',folder.resolve(),flush=True)
    return metrics,rankings,comparisons,folder




## Run a new experiment

Run from the Weikun folder. This creates a timestamped output folder and does not overwrite earlier results. Dependencies: numpy and pandas; matplotlib is optional for figures. The full comparison can take several minutes.


In [2]:
REPO = Path.cwd()
SOURCE = REPO / 'nj_daily_feature_outputs' / 'nj_daily_features.csv.gz'
OUTPUT = REPO / 'dynamic_panel_feature_experiment_outputs'
assert SOURCE.exists(), f'Open this notebook from the Weikun folder: {SOURCE}'
metrics, ranking, comparisons, RUN_FOLDER = run_feature_experiments(
    SOURCE, OUTPUT, bootstrap_repetitions=2000
)


Window 30: 8 models, 590 common regions, validation only.
  h=1: B0_basic fitted.
  h=1: H0_weekly_mean fitted.
  h=1: H1_history_extended fitted.
  h=1: D1_acceleration fitted.
  h=1: D2_jerk fitted.
  h=1: P0_basic_pressure7 fitted.
  h=1: P1_rich_pressure7 fitted.
  h=1: P2_rich_pressure14 fitted.
  h=2: B0_basic fitted.
  h=2: H0_weekly_mean fitted.
  h=2: H1_history_extended fitted.
  h=2: D1_acceleration fitted.
  h=2: D2_jerk fitted.
  h=2: P0_basic_pressure7 fitted.
  h=2: P1_rich_pressure7 fitted.
  h=2: P2_rich_pressure14 fitted.
  h=3: B0_basic fitted.
  h=3: H0_weekly_mean fitted.
  h=3: H1_history_extended fitted.
  h=3: D1_acceleration fitted.
  h=3: D2_jerk fitted.
  h=3: P0_basic_pressure7 fitted.
  h=3: P1_rich_pressure7 fitted.
  h=3: P2_rich_pressure14 fitted.
Window 30 completed; paired loss comparisons recorded.
Window 60: 8 models, 590 common regions, validation only.
  h=1: B0_basic fitted.
  h=1: H0_weekly_mean fitted.
  h=1: H1_history_extended fitted.
  h=1: D

## Inspect results

Read model_ranking.csv first, then paired_prediction_tests.csv and quarterly results. Choose features using error size, uncertainty and stability together. A small p-value alone is insufficient. Model/window selection on these validation results must remain labeled exploratory.


In [3]:
display(ranking)
display(comparisons.loc[(comparisons.window == 30) & (comparisons.horizon == 'overall')])
display(pd.read_csv(RUN_FOLDER / 'metrics_by_quarter.csv'))
print('Saved to:', RUN_FOLDER.resolve())


,window,model,mean_MAE,mean_RMSE,mean_bias,rank_within_window
3,30,H0_weekly_mean,1.096835,1.913533,-0.022630,1
1,30,D1_acceleration,1.097616,1.914579,-0.019961,2
2,30,D2_jerk,1.097655,1.914764,-0.020259,3
4,30,H1_history_extended,1.097805,1.915039,-0.019633,4
6,30,P1_rich_pressure7,1.101561,1.902110,0.042404,5
7,30,P2_rich_pressure14,1.106624,1.908184,0.042088,6
5,30,P0_basic_pressure7,1.127330,1.950848,0.049593,7
0,30,B0_basic,1.133520,1.987143,-0.042712,8
11,60,H0_weekly_mean,1.088668,1.891208,0.001468,1
14,60,P1_rich_pressure7,1.088990,1.880957,0.050807,2


,window,comparison,baseline,candidate,horizon,dates,regions,baseline_MAE,candidate_MAE,delta_MAE_baseline_minus_candidate,...,p_HAC_bandwidth28,region_share_with_lower_MAE,inference_status,delta_MAE_ci95_low,delta_MAE_ci95_high,delta_mean_RMSE_ci95_low,delta_mean_RMSE_ci95_high,p_HAC_Holm_all_64,positive_MAE_improvement,Holm_0_05_and_positive
0,30,weekly_mean,B0_basic,H0_weekly_mean,overall,362,590,1.133520,1.096835,0.036685,...,0.001071,0.920339,exploratory_validation_reused_not_confirmatory,0.020227,0.055978,0.039991,0.110559,0.004679,True,True
4,30,lag14_and_volatility,H0_weekly_mean,H1_history_extended,overall,362,590,1.096835,1.097805,-0.000971,...,0.268284,0.422034,exploratory_validation_reused_not_confirmatory,-0.002298,0.000960,-0.003165,0.002136,1.000000,False,False
8,30,acceleration,H1_history_extended,D1_acceleration,overall,362,590,1.097805,1.097616,0.000189,...,0.099449,0.540678,exploratory_validation_reused_not_confirmatory,-0.000015,0.000431,0.000049,0.001027,1.000000,True,False
12,30,jerk,D1_acceleration,D2_jerk,overall,362,590,1.097616,1.097655,-0.000039,...,0.502566,0.489831,exploratory_validation_reused_not_confirmatory,-0.000117,0.000114,-0.000343,0.000258,1.000000,False,False
16,30,pressure7_basic,B0_basic,P0_basic_pressure7,overall,362,590,1.133520,1.127330,0.006190,...,0.808842,0.615254,exploratory_validation_reused_not_confirmatory,-0.023495,0.052874,0.022950,0.070521,1.000000,True,False
20,30,pressure7_rich,D2_jerk,P1_rich_pressure7,overall,362,590,1.097655,1.101561,-0.003906,...,0.826637,0.610169,exploratory_validation_reused_not_confirmatory,-0.027417,0.027986,0.002800,0.031868,1.000000,False,False
24,30,pressure14_rich,D2_jerk,P2_rich_pressure14,overall,362,590,1.097655,1.106624,-0.008969,...,0.621331,0.567797,exploratory_validation_reused_not_confirmatory,-0.035892,0.021164,-0.005901,0.021687,1.000000,False,False
28,30,pressure14_vs_pressure7,P1_rich_pressure7,P2_rich_pressure14,overall,362,590,1.101561,1.106624,-0.005063,...,0.246130,0.189831,exploratory_validation_reused_not_confirmatory,-0.015255,0.000520,-0.015052,-0.002867,1.000000,False,False


,window,model,horizon,quarter,rows,MAE,RMSE,bias
0,30,B0_basic,1,2021Q1,53100,1.497753,2.294948,0.050334
1,30,B0_basic,1,2021Q2,53690,0.729360,1.295755,0.369083
2,30,B0_basic,1,2021Q3,54280,0.848612,1.355302,-0.172898
3,30,B0_basic,1,2021Q4,52510,1.357933,2.409692,-0.389127
4,30,B0_basic,2,2021Q1,53100,1.506075,2.302467,0.054808
...,...,...,...,...,...,...,...,...
187,60,P2_rich_pressure14,2,2021Q4,52510,1.411537,2.501778,-0.135266
188,60,P2_rich_pressure14,3,2021Q1,53100,1.431961,2.173680,0.015393
189,60,P2_rich_pressure14,3,2021Q2,53690,0.643734,1.233332,0.242838
190,60,P2_rich_pressure14,3,2021Q3,54280,0.926185,1.373651,0.079494


Saved to: C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\dynamic_panel_feature_experiment_outputs\feature_ablation_20261026T143430935788Z


In [4]:
from pathlib import Path
import os

def make_report_figures(folder):
    folder=Path(folder)
    figs=folder/'figures'; figs.mkdir(exist_ok=True)
    os.environ.setdefault('MPLCONFIGDIR',str(figs/'_cache'))
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    import pandas as pd
    import numpy as np
    plt.rcParams.update({'font.size':11,'axes.spines.top':False,'axes.spines.right':False,'figure.facecolor':'white'})
    paths=[]
    def save(fig,name):
        for ext in ['png','svg']:
            p=figs/f'{name}.{ext}'
            fig.savefig(p,dpi=300,bbox_inches='tight')
            paths.append(p)
        plt.close(fig)
    ranking=pd.read_csv(folder/'model_ranking.csv')
    tests=pd.read_csv(folder/'paired_prediction_tests.csv',dtype={'horizon':str})
    quarter=pd.read_csv(folder/'metrics_by_quarter.csv')
    labels={'B0_basic':'Basic history','H0_weekly_mean':'+ 7-day average',
        'H1_history_extended':'+ lag 14 and volatility','D1_acceleration':'+ acceleration',
        'D2_jerk':'+ jerk','P0_basic_pressure7':'Basic + pressure 7d',
        'P1_rich_pressure7':'Rich local + pressure 7d','P2_rich_pressure14':'Rich local + pressure 14d'}
    fig,axes=plt.subplots(1,2,figsize=(15,5.3))
    for ax,w in zip(axes,[30,60]):
        q=ranking.loc[ranking.window.eq(w)].sort_values('mean_MAE',ascending=False)
        yy=np.arange(len(q)); colors=['#2563eb' if 'pressure' in m else '#14805e' if m=='H0_weekly_mean' else '#64748b' for m in q.model]
        ax.scatter(q.mean_MAE,yy,c=colors,s=65)
        ax.set_yticks(yy,[labels[m] for m in q.model])
        for y,value in zip(yy,q.mean_MAE):ax.annotate(f'{value:.4f}',(value,y),xytext=(7,0),textcoords='offset points',va='center',fontsize=9)
        ax.set_xlim(ranking.mean_MAE.min()-.012,ranking.mean_MAE.max()+.025)
        ax.set_title(f'{w}-day training window');ax.set_xlabel('Mean validation MAE (cases; lower is better)');ax.grid(axis='x',alpha=.2)
    fig.suptitle('Feature ablations on the same 590 NJ regions',fontsize=16)
    fig.tight_layout(rect=(0,0,1,.94));save(fig,'01_feature_mae_comparison')

    q=tests.loc[tests.window.eq(30)&tests.horizon.eq('overall')].reset_index(drop=True)
    fig,ax=plt.subplots(figsize=(12,6))
    for y,row in enumerate(q.itertuples()):
        color='#14805e' if row.delta_MAE_baseline_minus_candidate>0 else '#b45309'
        ax.hlines(y,row.delta_MAE_ci95_low,row.delta_MAE_ci95_high,color=color,lw=2)
        ax.plot(row.delta_MAE_baseline_minus_candidate,y,'o',color=color)
        ax.text(1.02,y,f'Holm p={row.p_HAC_Holm_all_64:.3g}',transform=ax.get_yaxis_transform(),va='center',fontsize=10)
    ax.set_yticks(range(len(q)),q.comparison.str.replace('_',' '));ax.invert_yaxis();ax.axvline(0,color='#64748b',ls='--')
    ax.set_xlabel('MAE reduction: baseline minus candidate (positive = improvement)')
    ax.set_title('30-day window: paired prediction improvements and pointwise 95% intervals')
    ax.grid(axis='x',alpha=.2)
    fig.text(.5,.01,'14-day moving-block bootstrap intervals; HAC p-values adjusted over 64 tests. Exploratory validation.',ha='center',fontsize=9)
    fig.tight_layout(rect=(0,.04,.84,1));save(fig,'02_paired_improvement_intervals')

    selected=['B0_basic','H0_weekly_mean','D2_jerk','P1_rich_pressure7']
    fig,axes=plt.subplots(1,2,figsize=(13,4.8),sharey=True)
    for ax,w in zip(axes,[30,60]):
        d=quarter.loc[quarter.window.eq(w)].groupby(['model','quarter'],as_index=False).MAE.mean()
        for model in selected:
            z=d.loc[d.model.eq(model)].sort_values('quarter')
            ax.plot(z.quarter,z.MAE,marker='o',label=labels[model])
        ax.set_title(f'{w}-day training window');ax.set_xlabel('Forecast-origin quarter');ax.grid(alpha=.2)
    axes[0].set_ylabel('Mean validation MAE across horizons (cases)');axes[1].legend(fontsize=9)
    fig.suptitle('Temporal stability: improvements need not hold in every quarter',fontsize=14)
    fig.tight_layout(rect=(0,0,1,.93));save(fig,'03_quarter_stability')

    coef=pd.read_csv(folder/'coefficients_exploratory.csv',parse_dates=['fit_date'])
    panels=[('D1_acceleration','case_acceleration','Acceleration'),('D2_jerk','case_jerk','Jerk'),('P1_rich_pressure7','nj_covid_pressure_7d','Pressure after rich local history')]
    fig,axes=plt.subplots(3,1,figsize=(12,8),sharex=True)
    for ax,(model,feature,title) in zip(axes,panels):
        z=coef.loc[coef.window.eq(30)&coef.horizon.eq(1)&coef.model.eq(model)&coef.feature.eq(feature)].sort_values('fit_date')
        ax.plot(z.fit_date,z.estimate,color='#2563eb')
        ax.fill_between(z.fit_date,z.ci95_low_approx,z.ci95_high_approx,color='#2563eb',alpha=.16)
        ax.axhline(0,color='gray',ls='--');ax.set_title(title,fontsize=11);ax.set_ylabel('Coefficient');ax.grid(alpha=.15)
    fig.suptitle('Exploratory coefficient stability: horizon 1, 30-day window',fontsize=14)
    fig.text(.5,.01,'Approximate two-way clustered intervals; dynamic FE bias is not corrected. Not causal evidence.',ha='center',fontsize=9)
    fig.autofmt_xdate();fig.tight_layout(rect=(0,.04,1,.95));save(fig,'04_coefficient_stability_exploratory')

    archive=np.load(folder/'predictions'/'window_30d.npz',allow_pickle=False)
    zips=archive['zip'];where=np.flatnonzero(zips=='08901')
    if len(where):
        i=int(where[0]);model_names=archive['models'].tolist();dates=archive['date']
        fig,ax=plt.subplots(figsize=(13,4.8))
        ax.plot(dates+np.timedelta64(1,'D'),archive['actual_cases'][0,i],color='#64748b',alpha=.65,lw=1,label='Actual recorded-onset cases')
        for model,color in [('B0_basic','#b45309'),('H0_weekly_mean','#14805e')]:
            ax.plot(dates+np.timedelta64(1,'D'),np.maximum(archive['raw_predictions'][model_names.index(model),0,i],0),color=color,lw=1.3,label=labels[model])
        ax.set_title('NJ 08901: one-day-ahead validation forecasts (30-day training window)')
        ax.set_xlabel('Target date');ax.set_ylabel('Recorded-onset cases');ax.legend(fontsize=9);ax.grid(alpha=.2)
        fig.autofmt_xdate();fig.tight_layout();save(fig,'05_example_zip_08901')
    return paths


In [5]:
try:
    paths = make_report_figures(RUN_FOLDER)
    print('Figures saved:', len(paths), 'in', RUN_FOLDER / 'figures')
except ImportError:
    print('Optional figures need matplotlib. Install it in this notebook kernel with %pip install matplotlib, then rerun this cell.')


Figures saved: 10 in C:\Users\41240\OneDrive\文档\GitHub\practicum-project\Weikun\dynamic_panel_feature_experiment_outputs\feature_ablation_20261026T143430935788Z\figures


## Reopen existing results without refitting

Use the cell below independently when you only want the already completed results. PNG files are 300 dpi for presentations; SVG files support scalable editing.


In [6]:
EXISTING = Path.cwd() / 'dynamic_panel_feature_experiment_outputs' / 'feature_ablation_20261026T142422577198Z'
display(pd.read_csv(EXISTING / 'model_ranking.csv'))
display(pd.read_csv(EXISTING / 'paired_prediction_tests.csv'))
print((EXISTING / 'RESULTS.md').read_text(encoding='utf-8'))


,window,model,mean_MAE,mean_RMSE,mean_bias,rank_within_window
0,30,H0_weekly_mean,1.096835,1.913533,-0.022630,1
1,30,D1_acceleration,1.097616,1.914579,-0.019961,2
2,30,D2_jerk,1.097655,1.914764,-0.020259,3
3,30,H1_history_extended,1.097805,1.915039,-0.019633,4
4,30,P1_rich_pressure7,1.101561,1.902110,0.042404,5
5,30,P2_rich_pressure14,1.106624,1.908184,0.042088,6
6,30,P0_basic_pressure7,1.127330,1.950848,0.049593,7
7,30,B0_basic,1.133520,1.987143,-0.042712,8
8,60,H0_weekly_mean,1.088668,1.891208,0.001468,1
9,60,P1_rich_pressure7,1.088990,1.880957,0.050807,2


,window,comparison,baseline,candidate,horizon,dates,regions,baseline_MAE,candidate_MAE,delta_MAE_baseline_minus_candidate,...,p_HAC_bandwidth28,region_share_with_lower_MAE,inference_status,delta_MAE_ci95_low,delta_MAE_ci95_high,delta_mean_RMSE_ci95_low,delta_mean_RMSE_ci95_high,p_HAC_Holm_all_64,positive_MAE_improvement,Holm_0_05_and_positive
0,30,weekly_mean,B0_basic,H0_weekly_mean,overall,362,590,1.133520,1.096835,0.036685,...,0.001071,0.920339,exploratory_validation_reused_not_confirmatory,0.020227,0.055978,0.039991,0.110559,0.004679,True,True
1,30,weekly_mean,B0_basic,H0_weekly_mean,1,362,590,1.105243,1.068575,0.036668,...,0.000399,0.857627,exploratory_validation_reused_not_confirmatory,0.021111,0.054550,0.042318,0.103309,0.001076,True,True
2,30,weekly_mean,B0_basic,H0_weekly_mean,2,362,590,1.136526,1.098398,0.038128,...,0.001323,0.918644,exploratory_validation_reused_not_confirmatory,0.020943,0.058427,0.041131,0.112821,0.005863,True,True
3,30,weekly_mean,B0_basic,H0_weekly_mean,3,362,590,1.158791,1.123531,0.035260,...,0.002541,0.888136,exploratory_validation_reused_not_confirmatory,0.018366,0.055327,0.034455,0.116469,0.018547,True,True
4,30,lag14_and_volatility,H0_weekly_mean,H1_history_extended,overall,362,590,1.096835,1.097805,-0.000971,...,0.268284,0.422034,exploratory_validation_reused_not_confirmatory,-0.002298,0.000960,-0.003165,0.002136,1.000000,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
59,60,pressure14_rich,D2_jerk,P2_rich_pressure14,3,362,590,1.119244,1.122163,-0.002919,...,0.872196,0.625424,exploratory_validation_reused_not_confirmatory,-0.027935,0.027547,-0.003940,0.016811,1.000000,False,False
60,60,pressure14_vs_pressure7,P1_rich_pressure7,P2_rich_pressure14,overall,362,590,1.088990,1.091891,-0.002901,...,0.455740,0.300000,exploratory_validation_reused_not_confirmatory,-0.010808,0.002623,-0.011798,-0.002907,1.000000,False,False
61,60,pressure14_vs_pressure7,P1_rich_pressure7,P2_rich_pressure14,1,362,590,1.059559,1.060917,-0.001359,...,0.678319,0.345763,exploratory_validation_reused_not_confirmatory,-0.007844,0.003284,-0.008688,-0.001989,1.000000,False,False
62,60,pressure14_vs_pressure7,P1_rich_pressure7,P2_rich_pressure14,2,362,590,1.089817,1.092593,-0.002776,...,0.473093,0.316949,exploratory_validation_reused_not_confirmatory,-0.010709,0.002648,-0.011758,-0.002957,1.000000,False,False


# NJ feature experiments: validation results

Mean MAE/RMSE average the three horizon-specific scores. Lower is better. This is exploratory validation, not final holdout inference.

| Window | Model | Mean MAE | Mean RMSE | Rank |
|---|---|---:|---:|---:|
| 30 | H0_weekly_mean | 1.096835 | 1.913533 | 1 |
| 30 | D1_acceleration | 1.097616 | 1.914579 | 2 |
| 30 | D2_jerk | 1.097655 | 1.914764 | 3 |
| 30 | H1_history_extended | 1.097805 | 1.915039 | 4 |
| 30 | P1_rich_pressure7 | 1.101561 | 1.902110 | 5 |
| 30 | P2_rich_pressure14 | 1.106624 | 1.908184 | 6 |
| 30 | P0_basic_pressure7 | 1.127330 | 1.950848 | 7 |
| 30 | B0_basic | 1.133520 | 1.987143 | 8 |
| 60 | H0_weekly_mean | 1.088668 | 1.891208 | 1 |
| 60 | P1_rich_pressure7 | 1.088990 | 1.880957 | 2 |
| 60 | D2_jerk | 1.090895 | 1.892660 | 3 |
| 60 | D1_acceleration | 1.090944 | 1.892841 | 4 |
| 60 | H1_history_extended | 1.091228 | 1.893485 | 5 |
| 60 | P2_rich_pressure14 | 1.091891 | 1.886296 | 6 |
| 60 | P0_basic_pressure7 | 1.1296

## Output guide

| File | Purpose |
|---|---|
| RESULTS.md / METHODS.md | Report summary, definitions and inference limitations |
| model_ranking.csv / metrics_by_model_horizon.csv | MAE, RMSE and bias by model/window/horizon |
| paired_prediction_tests.csv | Effect sizes, block CIs, raw and Holm-adjusted p-values |
| block_length_sensitivity.csv | Dependence sensitivity |
| coefficients_exploratory.csv | Every refit coefficient, clustered SE, interval and p-value |
| coefficient_stability.csv | Descriptive coefficient stability; no averaged p-values |
| metrics_by_quarter.csv / metrics_by_region.csv | Temporal and regional variation |
| training_log.csv / experiment_protocol.json | Timing checks, specifications and source hash |
| predictions/*.npz | Compressed raw predictions and observed outcomes |
| figures/ | Report-ready PNG and SVG figures |

Load prediction archives with numpy.load(path, allow_pickle=False). raw_predictions has model × horizon × region × forecast-date axes; actual_cases has horizon × region × forecast-date axes. Scoring uses maximum(raw_predictions, 0). Dates in the archive are forecast origins; target dates add the horizon.
